# Prompt-tuning for extracting relationship triplets

## Evaluation

- formation sample (text -> strat_name (formations)): really bias, need update


## Steps

1. Create entry prompt (V1)
1. Use LLM to generate relationship triplets
1. Try different models (GPT-4-turbo, Claude3-Opus, Mixtral)
1. Manually example and update prompt manually.

Further improvements possibilities: 

- Can also use LLM to re-generate prompt
- Can use genetic algorithm to mix and match prompt
- Can use LLM to "reason" and and improve prompt (e.g., Given the result is missing A, B, C, and the task is XYZ, can you come up with a better task description?)
- Can collect successful example for few-shot learning 

In [1]:
import logging
from typing import Any, Callable

import pandas as pd
from evals.graph_dataset.formation_extraction import eval as get_eval
from tqdm import tqdm

from text2graph.llm import AnthropicModel, OpenAIModel, OpenSourceModel, llm_graph

logging.basicConfig(level=logging.INFO)

In [2]:
# To provide some resolution in the lower end of the metric, we can use a more lenient metric


def fuzzy_match(a: str, b: str) -> bool:
    """Return True if a and b are similar."""

    intersect_score = a.lower() in b.lower() or b.lower() in a.lower()
    # Remove formation
    a = a.replace("Formation", "").strip()
    b = b.replace("Formation", "").strip()

    loose_intersect_score = a.lower() in b.lower() or b.lower() in a.lower()
    return intersect_score or loose_intersect_score


def get_metrics(y_true: list[str], y_pred: list[str]) -> dict:
    """Calculate the metrics for the output."""

    print(f"{y_true=}", f"{y_pred=}")
    tp = len(set(y_true) & set(y_pred))
    fp = len(set(y_pred) - set(y_true))
    fn = len(set(y_true) - set(y_pred))
    if tp == 0:
        precision = 0
        recall = 0
        f1 = 0
    else:
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        f1 = 2 * (precision * recall) / (precision + recall)

    fuzzy_tp = sum(fuzzy_match(a, b) for a in y_true for b in y_pred)
    return {"precision": precision, "recall": recall, "f1": f1, "fuzzy_tp": fuzzy_tp}



For now, each prompt x model combination will need a post-processor

In [3]:
def run(
    model: OpenSourceModel | AnthropicModel | OpenAIModel,
    prompt_version: str,
    post_process_fn: Callable,
    n: int,
    save: bool = True,
) -> pd.DataFrame:
    """Run the model on the eval dataset."""

    eval = get_eval(subset=n, data="tmp/formation_sample.parquet.gzip")

    raw_outputs = []
    for text in tqdm(eval.x.tolist()):
        raw = llm_graph(text=text, model=model, prompt_version=prompt_version)
        raw_outputs.append(raw)

    y_pred = [post_process_fn(y) for y in raw_outputs]  # list[list[str]]
    y_pred_flat = [", ".join(y) for y in y_pred]  # list[str]

    eval_summary = eval.from_predictions(y_pred_flat)

    # Item level metrics
    df = pd.DataFrame(
        [get_metrics([y_true], y_pred) for y_true, y_pred in zip(eval.y_true, y_pred)]
    )
    df["text"] = eval.x.tolist()
    df["label"] = eval.y_true.tolist()
    df["raw_output"] = raw_outputs
    df["processed_output"] = y_pred
    df["model"] = model.value
    df["prompt_version"] = prompt_version
    df["post_process_fn"] = post_process_fn.__name__
    df["macro_f1"] = eval_summary.metrics["macro_f1"]
    df["macro_exact_match"] = eval_summary.metrics["exact_match"]

    # Prevent saving error
    str_cols = ["text", "label", "raw_output", "processed_output"]
    cat_cols = ["model", "prompt_version", "post_process_fn"]
    df = df.astype({col: str for col in str_cols})
    df = df.astype({col: 'category' for col in cat_cols})

    if save:
        df.to_parquet(f"evals/{model.value}_{prompt_version}_{n}.parquet.gzip")
    return df

### GPT baseline

In [ ]:
def v1_post_process_gpt(y: list[dict[str, Any]]) -> list[str]:
    """Post-process an output from v1 prompt."""

    strat_names = []

    if "locations" not in y:
        return []

    for location in y["locations"]:
        if "stratigraphic_units" in location:
            strat_names.extend(location["stratigraphic_units"])
    return strat_names

run(
    model=OpenAIModel.GPT4T,
    prompt_version="v1",
    post_process_fn=v1_post_process_gpt,
    n=30,
    save=True
)


### Anthropic

In [ ]:
def v1_post_process_anthropic(y: list[dict[str, Any]]) -> list[str]:
    """Post-process an output from v1 prompt."""

    strat_names = []
    for location in y:
        if "stratigraphic_units" in location:
            strat_names.extend(location["stratigraphic_units"])
    return strat_names

df = run(
    model=AnthropicModel.CLAUDE3OPUS,
    prompt_version="v1",
    post_process_fn=v1_post_process_anthropic,
    n=30,
    save=False
)


In [ ]:
df["raw_output"] = df["raw_output"].astype(str)
df.to_parquet("evals/anthropic_claude3opus_v1_30.parquet.gzip")